The Preprocssing Pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

#1 Download a subset of the dataset

#2 Load the first 50,000 rows. This ensures we stay within Google Colab's free RAM limits while exceeding the 5,000 row requirement
df = pd.read_csv('traffic_data.csv', nrows=50000)

#3 Deep Cleaning: The 'Industry Standard' Routine
def clean_dataset(df):
  # Remove whitespace from column names
  df.columns = df.columns.str.strip()

  # Replace infinity or very large numbers with NaN, them drop them
  # Neural networks cannot compute gradients with 'inf' values
  df.replace([np.inf, -np.inf], np.nan, inplace=True)
  df.dropna(inplace=True)

  # Drop columns that are constants (zero variance) - they provide no predictive power
  df = df.loc [:, (df != df.iloc[0].any())]

  return df

df = clean_dataset(df)
#4 Verification of Research Constraints
print(f"--- Dataset Audit ---")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Features: {df.shape[1]}")
print(f"Target Class Distribution:\n{df['Label'].value_counts()}")
print(f"------------------------\n")

#5 Visualising the 'Needle in the Haystack'
plt.figure(figsize=(10, 6))
sns.countplot(x='Label', data=df, palette='viridis')
plt.title('Class Distribution: Benign vs Infiltration Attacks')
plt.yscale('log') # Log scale because attacks are rare
plt.show()

The DNN Architecture (Keras / TensorFlow)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_ids_model (input_dim, num_classes):
  model = models.Sequential([
      # Input layer
      layers.Input(shape=(input_dim,)),

      # Hidden Layer 1: Capturing broad traffic patterns
      layers.Dense(128, activation='relu'),
      layeres.BatchNormalization(),
      layers.Dropout(0.3),

      # Hidden Layer 2: Deepning the abstraction
      layers.Dense(64, activation='relu'),
      layers.BatchNormalization(),
      layers.Dropout(0.3),

      # Hidden Layer 3: Refining features
      layers.Dense(32, activation='relu'),

      # Output Layer: Softmax for multi-class or Sigmoid for binary
      layers.Dense(num_classes, activation='softmax' if num_classes > 2 else 'sigmoid')
  ])

  model.compile (
      optimizer='adam',
      loss='sparse categorical_crossentropy' if num_classes > 2 else 'binary_crossentropy',
      metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
  )
  return model

  # Calculate class weights to handle 'Infiltration' rarity
  from sklearn.utils import class_weight
  import numpy as np

  weights = class_weight.compute_class_weight (
      class_weight='balanced',
      classes=np.unique(y_train),
      y=y_train
  )
  class_weights_dict = dict(enumerate(weights))

  # Build and Summary
  model = build_ids_model(X_train.shape[1], len(np.unique(y_train)))
  model.summary()

Training Implementation Code

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1 Split the data (80% Train, 20% Validation or Test)
X_train, X_test, y_train, y_test = train_test = train_test_split(df.drop('Label', axis=1), df['Label'], test_size=0.2, random_state=42)

# 2 Scale the features (Essential for Neural Networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

# 3 Define our Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='best_ids_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1

    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=0.00001,
        verbose=1
    )
]

# 4 Start the training on the T4 GPU
print ("Starting training... monitoring Validation Loss to prevent overfitting. ")
history = model.fit(
    X_train, y_train,
    epochs=100, # We set a high number, but EarlyStopping will likely halt it earlier
    batch_size=1024,  # Large batch size to leverage GPU paralization
    validation_split=0.2, # Use 20% of training data for real-time monitoring
    class_weight=class_weights_dict,  # Using the weights we calculated earlier
    callbacks=callbacks,
    verbose=1
)

Evaluation Implementation Code

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, precision_recall_curve, auc

# 1 Generate predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int) # Standard 0.5 threshold

# 2 Detailed Classification Report
# This gives us Precision, Recall, and F1-Score for each class
print ("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Infiltration']))

# 3 Plot Confusion Matrix
fig, ax = plt.subplots (1, 2, figsize=(15, 6))

ConfusionMatrixDisplay.from_predictions (y_test, y_pred,
                                         display_labels=['Benign', 'Infiltration'],
                                         cmap = 'Blues', ax=ax[0])
ax[0].set_title('Confusion Matrix')

# 4 Plot Precision-Recall Curve
precision, recall, _ = precision_recall_curve (y_test, y_pred_prob)
pr_auc = auc(recall, precision)

ax[1].plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='red')
ax[1].set_xlabel('Recall (Detection Rate)')
ax[1].set_ylabel('Precision (Accuracy of Alarms)')
ax[1].set_title('Precision-Recall Curve')
ax[1].legend()

plt.tight_layout()
plt.show()

Implementing SHAP on the T4 GPU

In [ ]:
import shap

# 1 Select a small background distribution to represent 'normal'
# SHAP needs a baseline to compare predictions against
background = X_train[np.random.choice(X_train.shape[0], 100, replace=False)]

# 2 Initialise the Explainer
explainer = shap.DeepExplainer (model, background)

# 3 Explain predictions on a subset of the test set
# We take 50 samples to keep the computation fast on Colab
test_samples = X_test[np.random.choice(X_test.shape[0], 50, replace=False)]
shap_values = explainer.shap_values(test_samples)

# 4 Plot the Global Feature Importance
# This shows which features generally drive 'Infiltration' detections

shap.summary_plot(shap_values, test_samples, feature_names=df.drop('Label', axis=1).columns)

Python Visualisation using Matplotlib, Seaborn, and Scikit-Learn

In [ ]:
import numpy as np
import matplotlib.pylot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc, average_precision_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1 Set up Aesthetics
sns.set_theme (style='whitegrid')
plt.rcParams ['font.family'] = 'serif'

def plot_rare_event_analysis():
  # Generate synthetic imbalanced data (99% benign, 1% attack)
  X, y = make_classification (n_samples=1000, n_features=20,
                              n_informative=2, n_redundant=10,
                              n_clusters_per_class=1, weights=[0.99],
                              flip_y=0, random_state=42)

  X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.3, random_state=42)

  # Model A: Baseline (Unbalanced Weights)
  model_a = LogisticRegression()
  model_a.fit (X_train, y_train)
  y_score_a = model_a.predict_proba (X_test) [:, 1]
  y_pred_a = model_a.predict (X_test)

  # Model B: Optimised (Cost-Sensitive/Balanced Weights)
  model_b = RandomForestClassifier (class_weight='balanced', n_estimators=100)
  model_b.fit (X_train, y_train)
  y_score_b = model_b.predict_proba (X_test) [:, 1]
  y_pred_b = model.b.predict (X_test)

  # --- Figure 1: Confusion Matrices ---
  fig, ax = plt.subplots (1, 2, figsize=(14, 5))

  for i (pred, title) in enumerate ([(y_pred_a, 'Baseline Model'), (y_pred_b, 'Optimised Model (Balanced)')]):
    cm = confusion_matrix (y_test, pred)
    sns.heatmap (cm, annot=True, fmt='d', cmap='Blues', ax=ax[i], cbar=False)
    ax[i].set_title (f'Confusion Matrix: {title}', fontsize=14)
    ax[i].set_xlabel ('Predicted Label')
    ax[i].set_ylabel ('True Label')
    ax[i].set_xticklabels (['Benign', 'Attack'])
    ax[i].set_yticklabels (['Bening', 'Attack'])

  plt.tight_layout()
  plt.show()

  # --- Figure 2: Precision-Recall Curves ---
  plt.figure (figsize=(8, 6))

  for score_label in [(y_score_a, 'Baseline (LR'), (y_score_b, 'Optimised (RF-Balanced)')]:
    precision, recall, _ = precision_recall_curve (y_test, score)
    ap = average_precision_score (y_test, score)
    plt.plot (recall, precision, lw=2, label=f'{label} (AP = {ap:.2f})')

  plt.xlabel ('Recall (Sensitivity)')
  plt.ylabel ('Precision')
  plt.title ('Precision-Recall Curve for Rare-Event Detection', fontsize=14)
  plt.legend (loc="best")
  plt.show()

if __name__ == "__main__":
  plot_rare_event_analysis()
